# MODUL PRAKTIKUM DATA MINING
## Association Rule Mining (Market Basket Analysis)

Notebook ini merupakan file terpisah untuk mengerjakan **Tugas 2 (Bagian 5.2)** pada Modul Praktikum Data Mining – Association Rule Mining (Market Basket Analysis), 2026.

### Dataset yang digunakan
Dataset transaksi yang digunakan tetap sama dengan notebook praktikum utama, yaitu `transactions_arm_praktikum.csv`, berisi 60 transaksi dengan 12 jenis item.

### Alur pengerjaan
1. Memilih dua aturan asosiasi yang belum dibahas pada Bagian 4.6.
2. Menghitung nilai Leverage dan Conviction untuk kedua aturan.
3. Membandingkan nilai Leverage dan Conviction dengan Lift.
4. Memberikan interpretasi bisnis berdasarkan aturan asosiasi yang dipilih.

Seluruh proses menggunakan algoritma Apriori dengan `min_support = 0.15` dan `min_confidence = 0.5`, sesuai dengan pengaturan pada Bagian 4.3–4.4.

## Persiapan Data dan Pustaka

Tahap persiapan dilakukan dengan mengimpor pustaka yang dibutuhkan dan membaca dataset transaksi yang sama dengan praktikum utama. Data transaksi kemudian diubah menjadi matriks one-hot agar dapat diproses menggunakan algoritma Apriori.

In [1]:
%pip install -q mlxtend

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# Membaca dataset transaksi mentah
data = pd.read_csv('transactions_arm_praktikum.csv')

# transactions: list berisi transaksi (daftar item per transaksi)
transactions = [items.split(', ') for items in data['Items']]

print('Jumlah transaksi:', len(transactions))
for i, t in enumerate(transactions[:5], 1):
    print(f'T{i}: {t}')

Jumlah transaksi: 60
T1: ['Gula', 'Kopi', 'Susu']
T2: ['Gula', 'Kopi', 'Susu']
T3: ['Mentega', 'Roti', 'Susu', 'Telur']
T4: ['Mie Instan']
T5: ['Keju', 'Roti', 'Susu', 'Telur']


In [3]:
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_ary, columns=te.columns_)

print('Ukuran matriks one-hot:', df.shape)
print(df.iloc[:5, :6])

Ukuran matriks one-hot: (60, 12)
    Gula  Kecap   Keju   Kopi  Mentega  Mie Instan
0   True  False  False   True    False       False
1   True  False  False   True    False       False
2  False  False  False  False     True       False
3  False  False  False  False    False        True
4  False  False   True  False    False       False


## Pencarian Frequent Itemset dan Aturan Asosiasi

Pada tahap ini, algoritma Apriori digunakan untuk menemukan frequent itemset dengan `min_support = 0.15`. Selanjutnya, aturan asosiasi dibentuk menggunakan `min_confidence = 0.5`, sesuai dengan hasil Bagian 4.3–4.4.

Aturan yang dihasilkan diurutkan berdasarkan nilai Lift untuk memudahkan proses pemilihan aturan yang akan dianalisis.

In [4]:
MIN_SUP = 0.15
MIN_CONF = 0.5

freq_apriori = apriori(
    df,
    min_support=MIN_SUP,
    use_colnames=True
)

rules = association_rules(
    freq_apriori,
    metric='confidence',
    min_threshold=MIN_CONF
)

rules = rules.sort_values(
    'lift',
    ascending=False
).reset_index(drop=True)

print('Jumlah frequent itemset:', len(freq_apriori))
print('Jumlah aturan asosiasi:', len(rules))

Jumlah frequent itemset: 17
Jumlah aturan asosiasi: 18


## Pemilihan Aturan Asosiasi

Berdasarkan hasil aturan asosiasi pada Bagian 4.4, dipilih dua aturan yang belum dibahas pada Bagian 4.6, yaitu:

1. Mie Instan → Telur
2. Roti → Susu

Kedua aturan tersebut dipilih karena menunjukkan pola pembelian bersama yang dapat dianalisis lebih lanjut menggunakan nilai Leverage dan Conviction.

In [5]:
# Memilih dua aturan yang akan dianalisis
aturan_pilihan = [
    (frozenset(['Mie Instan']), frozenset(['Telur'])),
    (frozenset(['Roti']), frozenset(['Susu']))
]

selected_rules = rules[
    rules.apply(
        lambda row: (
            row['antecedents'],
            row['consequents']
        ) in aturan_pilihan,
        axis=1
    )
].copy()

cols_detail = [
    'antecedents',
    'consequents',
    'support',
    'confidence',
    'lift',
    'leverage',
    'conviction'
]

print('Dua aturan asosiasi yang dipilih:')
print(selected_rules[cols_detail].round(3).to_string(index=False))

Dua aturan asosiasi yang dipilih:
 antecedents consequents  support  confidence  lift  leverage  conviction
      (Roti)      (Susu)    0.217       0.722 2.281     0.122       2.460
(Mie Instan)     (Telur)    0.167       0.556 2.222     0.092       1.688


## Perhitungan Leverage dan Conviction

Leverage dan Conviction digunakan untuk menganalisis kekuatan aturan asosiasi secara lebih mendalam.

Rumus yang digunakan:

**1. Leverage**

Leverage mengukur selisih antara support gabungan aktual dengan support yang diharapkan apabila kedua itemset saling independen.

Leverage(X → Y) = Support(X ∪ Y) − Support(X) × Support(Y)

**2. Conviction**

Conviction mengukur kekuatan implikasi suatu aturan berdasarkan kemungkinan terjadinya kesalahan prediksi.

Conviction(X → Y) = (1 − Support(Y)) / (1 − Confidence(X → Y))

**3. Lift**

Lift digunakan sebagai pembanding untuk mengetahui kekuatan hubungan antara antecedent dan consequent.

Lift(X → Y) = Confidence(X → Y) / Support(Y)

In [6]:
hasil_perhitungan = []

for antecedent, consequent in aturan_pilihan:

    support_x = df[list(antecedent)].all(axis=1).mean()
    support_y = df[list(consequent)].all(axis=1).mean()

    support_xy = df[
        list(antecedent | consequent)
    ].all(axis=1).mean()

    confidence = support_xy / support_x
    lift = confidence / support_y

    leverage = support_xy - (support_x * support_y)

    conviction = (
        (1 - support_y) / (1 - confidence)
    )

    hasil_perhitungan.append({
        'Aturan': f"{', '.join(antecedent)} → {', '.join(consequent)}",
        'Support X': support_x,
        'Support Y': support_y,
        'Support X∪Y': support_xy,
        'Confidence': confidence,
        'Lift': lift,
        'Leverage': leverage,
        'Conviction': conviction
    })

hasil_df = pd.DataFrame(hasil_perhitungan)

print(hasil_df.round(3).to_string(index=False))

            Aturan  Support X  Support Y  Support X∪Y  Confidence  Lift  Leverage  Conviction
Mie Instan → Telur        0.3      0.250        0.167       0.556 2.222     0.092       1.688
       Roti → Susu        0.3      0.317        0.217       0.722 2.281     0.122       2.460


**Analisis hasil:**

Berdasarkan hasil perhitungan, aturan Mie Instan → Telur memiliki nilai Leverage sebesar 0.092 dan Conviction sebesar 1.688. Nilai Leverage positif menunjukkan bahwa kedua item lebih sering dibeli bersamaan dibandingkan dengan kondisi independen. Sementara itu, nilai Conviction di atas 1 menunjukkan adanya hubungan implikasi positif antara pembelian Mie Instan dan Telur.

Pada aturan Roti → Susu, diperoleh nilai Leverage sebesar 0.122 dan Conviction sebesar 2.460. Nilai tersebut menunjukkan bahwa pembelian Roti dan Susu secara bersamaan memiliki selisih kemunculan aktual yang lebih besar dibandingkan kondisi independen. Nilai Conviction yang lebih tinggi juga menunjukkan bahwa aturan Roti → Susu memiliki kekuatan implikasi yang lebih besar dibandingkan aturan Mie Instan → Telur.

Jika dibandingkan dengan Lift, kedua aturan memiliki nilai di atas 1, yaitu 2.222 dan 2.281. Hal ini menunjukkan adanya hubungan positif antara item yang dibeli. Aturan Roti → Susu memiliki nilai Lift, Leverage, dan Conviction lebih tinggi dibandingkan aturan Mie Instan → Telur.

## Interpretasi Bisnis

Berdasarkan dua aturan asosiasi yang telah dianalisis, diperoleh beberapa rekomendasi bisnis sebagai berikut.

**1. Mie Instan → Telur**

Aturan ini memiliki nilai Support sebesar 0.167, Confidence sebesar 0.556, dan Lift sebesar 2.222. Artinya, sebanyak 55.6% transaksi yang membeli Mie Instan juga membeli Telur. Nilai Lift menunjukkan bahwa pembelian Telur pada transaksi yang membeli Mie Instan sekitar 2.222 kali lebih tinggi dibandingkan tingkat pembelian Telur secara umum.

Berdasarkan pola tersebut, toko dapat menempatkan produk Mie Instan dan Telur pada rak yang berdekatan. Selain itu, toko dapat menawarkan paket bundling Mie Instan dan Telur untuk mendorong pembelian kedua produk dalam satu transaksi.

**2. Roti → Susu**

Aturan ini memiliki nilai Support sebesar 0.217, Confidence sebesar 0.722, dan Lift sebesar 2.281. Artinya, sebanyak 72.2% transaksi yang membeli Roti juga membeli Susu. Nilai Lift menunjukkan bahwa pembelian Susu pada transaksi yang membeli Roti sekitar 2.281 kali lebih tinggi dibandingkan tingkat pembelian Susu secara umum.

Berdasarkan pola tersebut, toko dapat menempatkan Roti dan Susu pada area yang berdekatan serta menawarkan paket bundling sarapan. Toko juga dapat menerapkan rekomendasi produk Susu ketika pelanggan memilih Roti, khususnya melalui sistem rekomendasi pada aplikasi belanja online.

## Kesimpulan

Berdasarkan analisis dua aturan asosiasi yang belum dibahas pada Bagian 4.6, yaitu Mie Instan → Telur dan Roti → Susu, diperoleh hasil bahwa kedua aturan memiliki nilai Leverage positif dan Conviction lebih dari 1. Hal ini menunjukkan adanya hubungan positif antara produk yang dibeli secara bersamaan.

Aturan Roti → Susu memiliki nilai Lift sebesar 2.281, Leverage sebesar 0.122, dan Conviction sebesar 2.460. Sementara itu, aturan Mie Instan → Telur memiliki nilai Lift sebesar 2.222, Leverage sebesar 0.092, dan Conviction sebesar 1.688.

Dengan demikian, aturan Roti → Susu menunjukkan hubungan asosiasi yang lebih kuat berdasarkan ketiga ukuran tersebut. Hasil analisis dapat dimanfaatkan sebagai dasar rekomendasi penataan rak, bundling produk, dan cross-selling. Namun, aturan asosiasi hanya menunjukkan hubungan antarproduk dan tidak membuktikan hubungan sebab-akibat.